In [1]:
!pip install imagecodecs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.5/24.5 MB 75.3 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import torch
import zipfile
from pathlib import Path
from tqdm import tqdm
import tifffile
import tempfile
from sklearn.model_selection import train_test_split
import shutil

class KagglePatchCreator:
    def __init__(self, 
                 input_image_dir,
                 input_mask_dir,
                 output_dir,
                 patch_size=64,
                 stride=64,
                 max_output_size_gb=19.0):
        """
        Creates patches for Kaggle with size constraints
        
        Args:
            input_image_dir: Directory with original images
            input_mask_dir: Directory with corresponding masks
            output_dir: Where to save the Kaggle-ready dataset
            patch_size: Size of patches (64,64,64)
            stride: Stride for sliding window
            max_output_size_gb: Maximum size in GB (Kaggle limit is ~20GB)
        """
        self.image_dir = Path(input_image_dir)
        self.mask_dir = Path(input_mask_dir)
        self.output_dir = Path(output_dir)
        self.patch_size = patch_size
        self.stride = stride
        self.max_output_size_bytes = max_output_size_gb * 1024**3
        
        # Create output directories
        self.output_images_dir = self.output_dir / 'train_images'
        self.output_masks_dir = self.output_dir / 'train_labels'
        self.output_images_dir.mkdir(parents=True, exist_ok=True)
        self.output_masks_dir.mkdir(parents=True, exist_ok=True)
        
    def get_volume_files(self):
        """Get all volume files from directories"""
        image_files = sorted(list(self.image_dir.glob('*')))
        mask_files = sorted(list(self.mask_dir.glob('*')))
        
        # Verify files match
        assert len(image_files) == len(mask_files), "Number of images and masks don't match"
        
        # Match by ID
        matched_files = []
        for img_file in image_files:
            mask_file = self.mask_dir / img_file.name
            if mask_file.exists():
                matched_files.append((img_file, mask_file))
            else:
                print(f"Warning: No mask found for {img_file.name}")

        train, test = train_test_split(matched_files, test_size=0.5, shuffle=True, random_state=42)
        
        return train
    
    def extract_patches_from_volume(self, volume, patch_size, stride):
        """Extract patches from a 3D volume"""
        patches = []
        positions = []
        
        depth, height, width = volume.shape
        
        # Calculate number of patches in each dimension
        depth_steps = (depth - patch_size) // stride + 1
        height_steps = (height - patch_size) // stride + 1
        width_steps = (width - patch_size) // stride + 1
        
        for d in range(depth_steps):
            for h in range(height_steps):
                for w in range(width_steps):
                    start_d = d * stride
                    start_h = h * stride
                    start_w = w * stride
                    
                    patch = volume[start_d:start_d+patch_size,
                                  start_h:start_h+patch_size,
                                  start_w:start_w+patch_size]
                    
                    patches.append(patch)
                    positions.append((start_d, start_h, start_w))
        
        return patches, positions
    
    def estimate_output_size(self, volume_pairs):
        """Estimate total output size to stay under Kaggle limit"""
        total_patches = 0
        sample_size = 0
        
        for img_file, mask_file in volume_pairs[:1]:  # Sample first file
            # Load one volume to estimate
            img = tifffile.imread(str(img_file))
            mask = tifffile.imread(str(mask_file))
            
            # Get patch info
            img_patches, _ = self.extract_patches_from_volume(
                img, self.patch_size, self.stride
            )
            
            # Estimate size per patch (float32 for images, uint8 for masks)
            img_patch_size_bytes = img_patches[0].size * 4  # float32 = 4 bytes
            mask_patch_size_bytes = img_patches[0].size * 1  # uint8 = 1 byte
            
            sample_size = len(img_patches) * (img_patch_size_bytes + mask_patch_size_bytes)
            total_patches += len(img_patches)
            
            print(f"Sample: {img_file.name}")
            print(f"  Original size: {img.shape}")
            print(f"  Patches per volume: {len(img_patches)}")
            print(f"  Estimated size per volume: {sample_size/1024**2:.2f} MB")
            
        total_volumes = len(volume_pairs)
        total_estimated_size = sample_size * total_volumes
        
        print(f"\nTotal estimation:")
        print(f"  Volumes: {total_volumes}")
        print(f"  Total patches: {total_patches * total_volumes}")
        print(f"  Estimated dataset size: {total_estimated_size/1024**3:.2f} GB")
        
        if total_estimated_size > self.max_output_size_bytes:
            print(f"\n⚠️  Warning: Estimated size ({total_estimated_size/1024**3:.2f} GB) "
                  f"exceeds Kaggle limit ({self.max_output_size_gb} GB)")
            print("Consider increasing stride or filtering volumes")
            
        return total_estimated_size
    
    def create_patches(self, volume_pairs, max_patches_per_volume=None):
        """Create and save patches for all volumes"""
        patch_counter = 0
        volume_counter = 0
        
        for img_file, mask_file in tqdm(volume_pairs, desc="Processing volumes"):
            try:
                # Load volume and mask
                img_volume = tifffile.imread(str(img_file)).astype(np.uint8)
                mask_volume = tifffile.imread(str(mask_file)).astype(np.uint8)
                
                # Get original shape
                original_shape = img_volume.shape
                
                # Extract patches
                img_patches, positions = self.extract_patches_from_volume(
                    img_volume, self.patch_size, self.stride
                )
                mask_patches, _ = self.extract_patches_from_volume(
                    mask_volume, self.patch_size, self.stride
                )
                
                # Limit patches per volume if needed
                if max_patches_per_volume and len(img_patches) > max_patches_per_volume:
                    indices = np.random.choice(len(img_patches), max_patches_per_volume, replace=False)
                    img_patches = [img_patches[i] for i in indices]
                    mask_patches = [mask_patches[i] for i in indices]
                    positions = [positions[i] for i in indices]
                
                # Save each patch
                for i, (img_patch, mask_patch, pos) in enumerate(zip(img_patches, mask_patches, positions)):
                    # Create unique patch name
                    vol_id = img_file.stem
                    patch_name = f"{vol_id}_{original_shape[0]}_patch{pos}"
                    
                    # Save image patch
                    img_path = self.output_images_dir / f"{patch_name}.tif"
                    tifffile.imwrite(str(img_path), img_patch, dtype=np.uint8, compression='ZSTD')
                    
                    # Save mask patch
                    mask_path = self.output_masks_dir / f"{patch_name}.tif"
                    tifffile.imwrite(str(mask_path), mask_patch, dtype=np.uint8, compression='ZSTD')
                    
                    patch_counter += 1
                
                volume_counter += 1
                print(f"\nProcessed {volume_counter}/{len(volume_pairs)}: {vol_id}")
                print(f"  Shape: {original_shape}")
                print(f"  Patches extracted: {len(img_patches)}")
                print(f"  Total patches so far: {patch_counter}")
                
            except Exception as e:
                print(f"Error processing {img_file.name}: {e}")
                continue
        
        print(f"\n✅ Processing complete!")
        print(f"Total volumes processed: {volume_counter}")
        print(f"Total patches created: {patch_counter}")
        print(f"Images saved to: {self.output_images_dir}")
        print(f"Masks saved to: {self.output_masks_dir}")
        
        return patch_counter
    
    def create_kaggle_dataset(self, volume_pairs, zip_output=True):
        """Main method to create Kaggle-ready dataset"""
        print("="*60)
        print("Creating Kaggle Dataset")
        print("="*60)
        
        # Step 1: Estimate size
        estimated_size = self.estimate_output_size(volume_pairs)
        
        # Step 2: Ask for confirmation if size is large
        if estimated_size > self.max_output_size_bytes * 0.8:  # 80% of limit
            response = input(f"\nEstimated size is {estimated_size/1024**3:.2f} GB. "
                           f"Continue? (y/n): ")
            if response.lower() != 'y':
                print("Aborted by user")
                return
        
        # Step 3: Create patches
        total_patches = self.create_patches(volume_pairs)
        
        # Step 4: Create zip files if requested
        if zip_output:
            self.create_zip_files()
        
        # Step 5: Create dataset metadata
        self.create_metadata(total_patches, volume_pairs)
        
        print("\n" + "="*60)
        print("Dataset Creation Complete!")
        print("="*60)
    
    def create_zip_files(self):
        """Create zip files for Kaggle upload"""
        print("\nCreating zip files...")
        
        # Create zip for images
        images_zip_path = self.output_dir / 'train_images.zip'
        with zipfile.ZipFile(images_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file_path in tqdm(list(self.output_images_dir.glob('*.tiff')), 
                                 desc="Zipping images"):
                zipf.write(file_path, file_path.name)
        
        # Create zip for masks
        masks_zip_path = self.output_dir / 'train_labels.zip'
        with zipfile.ZipFile(masks_zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for file_path in tqdm(list(self.output_masks_dir.glob('*.tiff')), 
                                 desc="Zipping masks"):
                zipf.write(file_path, file_path.name)
        
        print(f"\nZip files created:")
        print(f"  Images: {images_zip_path} ({images_zip_path.stat().st_size/1024**3:.2f} GB)")
        print(f"  Masks: {masks_zip_path} ({masks_zip_path.stat().st_size/1024**3:.2f} GB)")
    
    def create_metadata(self, total_patches, volume_pairs):
        """Create metadata file for the dataset"""
        metadata = {
            "dataset_info": {
                "total_patches": total_patches,
                "patch_size": self.patch_size,
                "stride": self.stride,
                "original_volumes": len(volume_pairs),
                "output_directory": str(self.output_dir)
            },
            "volume_sizes": {},
            "file_structure": {
                "train_images": "Contains image patches as .tiff files",
                "train_labels": "Contains corresponding mask patches as .tiff files",
                "naming_convention": "id_originalDepthxoriginalHeightxoriginalWidth_patchXXXXXX.tiff"
            }
        }
        
        # Add original volume info
        for img_file, mask_file in volume_pairs:
            try:
                img = tifffile.imread(str(img_file))
                metadata["volume_sizes"][img_file.stem] = {
                    "original_shape": img.shape,
                    "image_file": img_file.name,
                    "mask_file": mask_file.name
                }
            except:
                pass
        
        # Save metadata
        import json
        metadata_path = self.output_dir / 'dataset_metadata.json'
        with open(metadata_path, 'w') as f:
            json.dump(metadata, f, indent=2)
        
        print(f"\nMetadata saved to: {metadata_path}")

In [3]:
# Main execution
if __name__ == "__main__":
    
    # Initialize patch creator
    creator = KagglePatchCreator(
        input_image_dir='/kaggle/input/vesuvius-challenge-surface-detection/train_images',
        input_mask_dir = '/kaggle/input/vesuvius-challenge-surface-detection/train_labels',
        output_dir='',
        patch_size=64,
        stride=64,
        max_output_size_gb=20
    )
    
    # Get volume files
    volume_pairs = creator.get_volume_files()
    print(f"Found {len(volume_pairs)} volume-mask pairs")
    
    # Create dataset
    creator.create_patches(volume_pairs)

Found 393 volume-mask pairs


Processing volumes:   0%|          | 1/393 [00:02<18:45,  2.87s/it]


Processed 1/393: 1542708535
  Shape: (320, 320, 320)
  Patches extracted: 125
  Total patches so far: 125


Processing volumes:   1%|          | 2/393 [00:05<18:16,  2.80s/it]


Processed 2/393: 1294570892
  Shape: (320, 320, 320)
  Patches extracted: 125
  Total patches so far: 250


...


Processed 391/393: 2306395636
  Shape: (320, 320, 320)
  Patches extracted: 125
  Total patches so far: 47319


Processing volumes: 100%|█████████▉| 392/393 [17:20<00:03,  3.17s/it]


Processed 392/393: 3144205538
  Shape: (320, 320, 320)
  Patches extracted: 125
  Total patches so far: 47444


Processing volumes: 100%|██████████| 393/393 [17:23<00:00,  2.66s/it]


Processed 393/393: 1522833038
  Shape: (320, 320, 320)
  Patches extracted: 125
  Total patches so far: 47569

✅ Processing complete!
Total volumes processed: 393
Total patches created: 47569
Images saved to: train_images
Masks saved to: train_labels
